In [1]:
from dotenv import load_dotenv; load_dotenv()
from langchain.tools import tool
from langchain.agents import create_agent

In [1]:
from paper_agent.services.papers import search_papers, add_paper, list_papers
from paper_agent.services.retrieval import retrieve

In [2]:
# @tool(response_format="content_and_artifact")
def retrieve_tool(query: str, arxiv_id: str) -> tuple:
    """Find the most relevant passages from an added paper. Use list_papers to get arxiv_id"""
    return retrieve(query, arxiv_id)

In [4]:
tools = [tool(search_papers), tool(add_paper), tool(list_papers), retrieve_tool]

In [5]:
system_prompt = """You are an assistant that helps users understand research papers from arXiv.

Tools:
- search_papers(title): lists candidate papers (arxiv_id, title, authors, year). Does not download.
- add_paper(arxiv_id): downloads and embeds ONE paper. Safe to call again; it skips work already done.
- list_papers(): lists papers already added, with their arxiv_id.
- retrieve_tool(query, arxiv_id): returns passages from one added paper.

Workflow:
1. When the user names a paper or topic, call search_papers and show the candidates
   (number, title, authors, year). Ask which one they want. Do NOT call add_paper until they choose.
2. After they choose, call add_paper with that paper's arxiv_id and confirm it is ready.
   If it was already embedded, just say so.
3. For questions about a paper, call list_papers if you don't know its arxiv_id, then call
   retrieve_tool with a focused query and that arxiv_id. If several papers are added and it is
   unclear which one the user means, ask.

Answering rules:
- Use ONLY the retrieved passages. Do not add facts, dates, or names from your own knowledge.
- Cite page numbers, e.g. (page 5).
- If the result is NO_RELEVANT_CONTEXT, or the passages don't answer the question, say the paper
  doesn't appear to cover it. Do not guess.
- A mention in a reference list or bibliography does not mean the paper discusses that topic.
- For a question that has several parts, you may call retrieve_tool more than once with different queries.
- Explain clearly and concisely. Write formulas in LaTeX.
"""

In [6]:
from IPython.display import Markdown, display

def mprint(text: str) -> None:
    display(Markdown(text))

In [7]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt=system_prompt
)

In [9]:
config = {"configurable": {"thread_id": "3"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="download attention is all you need paper")]},
    config=config
)

In [10]:
response

{'messages': [HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='6a847f5e-042e-40c8-92ef-7719b9b9b725'),
  HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='4f43e136-c0fc-4111-abc9-86199fe58607'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_papers', 'arguments': '{"title": "Attention Is All You Need"}'}, '__gemini_function_call_thought_signatures__': {'call_186349': 'EnMKcQFpFH0Tfk/doi4Mq8HdGcMWznBoehq5/o0L0AC5EYKGym6vz6qG+q/0w3B+uoWTg0mW4V7+Sc4KSyzDW7dQyWKwzBBJ+Oon53MH6VHBbQVklxnKugRYBlSSBbXC7/wxu9LP6b+udnE6jQGghO65GQLc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a18aaa-75a7-7527-8f17-4242bef2df72-0', tool_calls=[{'name': 'search_papers', 'args': {'title': 'Attention Is All You Need'}, 'id': 'call_186349', 'type': 'to

In [11]:
print(response["messages"][-1].content[0]["text"])

I found several papers with similar titles. The original "Attention Is All You Need" paper (Vaswani et al., 2017) is:

*   **1706.03762v7: Attention Is All You Need** by Ashish Vaswani, Noam Shazeer, Niki Parmar, et al. (2017)

Is this the one you would like me to download?


In [12]:
config = {"configurable": {"thread_id": "3"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Yes this one donwload ?")]},
    config=config
)

In [13]:
response

{'messages': [HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='6a847f5e-042e-40c8-92ef-7719b9b9b725'),
  HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='4f43e136-c0fc-4111-abc9-86199fe58607'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_papers', 'arguments': '{"title": "Attention Is All You Need"}'}, '__gemini_function_call_thought_signatures__': {'call_186349': 'EnMKcQFpFH0Tfk/doi4Mq8HdGcMWznBoehq5/o0L0AC5EYKGym6vz6qG+q/0w3B+uoWTg0mW4V7+Sc4KSyzDW7dQyWKwzBBJ+Oon53MH6VHBbQVklxnKugRYBlSSBbXC7/wxu9LP6b+udnE6jQGghO65GQLc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a18aaa-75a7-7527-8f17-4242bef2df72-0', tool_calls=[{'name': 'search_papers', 'args': {'title': 'Attention Is All You Need'}, 'id': 'call_186349', 'type': 'to

In [14]:
print(response["messages"][-1].content[0]["text"])

The paper "Attention Is All You Need" (1706.03762v7) has been successfully downloaded and is ready. How can I help you with it?


In [15]:
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="In the paper vaswani talk about BERT ?")]},
    config=config
)

In [16]:
response

{'messages': [HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='6a847f5e-042e-40c8-92ef-7719b9b9b725'),
  HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='4f43e136-c0fc-4111-abc9-86199fe58607'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_papers', 'arguments': '{"title": "Attention Is All You Need"}'}, '__gemini_function_call_thought_signatures__': {'call_186349': 'EnMKcQFpFH0Tfk/doi4Mq8HdGcMWznBoehq5/o0L0AC5EYKGym6vz6qG+q/0w3B+uoWTg0mW4V7+Sc4KSyzDW7dQyWKwzBBJ+Oon53MH6VHBbQVklxnKugRYBlSSBbXC7/wxu9LP6b+udnE6jQGghO65GQLc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a18aaa-75a7-7527-8f17-4242bef2df72-0', tool_calls=[{'name': 'search_papers', 'args': {'title': 'Attention Is All You Need'}, 'id': 'call_186349', 'type': 'to

In [17]:
print(response["messages"][-1].content[0]["text"])

The paper does not appear to discuss BERT.


In [19]:
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="how much attention head is used in paper or optimal ?")]},
    config=config
)

In [20]:
response

{'messages': [HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='6a847f5e-042e-40c8-92ef-7719b9b9b725'),
  HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='4f43e136-c0fc-4111-abc9-86199fe58607'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_papers', 'arguments': '{"title": "Attention Is All You Need"}'}, '__gemini_function_call_thought_signatures__': {'call_186349': 'EnMKcQFpFH0Tfk/doi4Mq8HdGcMWznBoehq5/o0L0AC5EYKGym6vz6qG+q/0w3B+uoWTg0mW4V7+Sc4KSyzDW7dQyWKwzBBJ+Oon53MH6VHBbQVklxnKugRYBlSSBbXC7/wxu9LP6b+udnE6jQGghO65GQLc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a18aaa-75a7-7527-8f17-4242bef2df72-0', tool_calls=[{'name': 'search_papers', 'args': {'title': 'Attention Is All You Need'}, 'id': 'call_186349', 'type': 'to

In [21]:
mprint(response["messages"][-1].content[0]["text"])

In the paper "Attention Is All You Need," the authors employ **$h = 8$ parallel attention layers, or heads** (page 5).

For each of these heads, they use a dimensionality of $d_k = d_v = d_{model}/h = 64$. The paper notes that because of the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality (page 5).

In [22]:
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="which optimizer they used for that ?")]},
    config=config
)

In [23]:
response

{'messages': [HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='6a847f5e-042e-40c8-92ef-7719b9b9b725'),
  HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='4f43e136-c0fc-4111-abc9-86199fe58607'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_papers', 'arguments': '{"title": "Attention Is All You Need"}'}, '__gemini_function_call_thought_signatures__': {'call_186349': 'EnMKcQFpFH0Tfk/doi4Mq8HdGcMWznBoehq5/o0L0AC5EYKGym6vz6qG+q/0w3B+uoWTg0mW4V7+Sc4KSyzDW7dQyWKwzBBJ+Oon53MH6VHBbQVklxnKugRYBlSSBbXC7/wxu9LP6b+udnE6jQGghO65GQLc'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a18aaa-75a7-7527-8f17-4242bef2df72-0', tool_calls=[{'name': 'search_papers', 'args': {'title': 'Attention Is All You Need'}, 'id': 'call_186349', 'type': 'to

In [24]:
mprint(response["messages"][-1].content[0]["text"])

The paper uses the **Adam optimizer** [20] with the following hyperparameters (page 7):

*   $\beta_1 = 0.9$
*   $\beta_2 = 0.98$
*   $\epsilon = 10^{-9}$

The authors also used a learning rate schedule that varied over the course of training according to the formula:

$$lrate = d_{model}^{-0.5} \cdot \min(\text{step\_num}^{-0.5}, \text{step\_num} \cdot \text{warmup\_steps}^{-1.5})$$

This involves increasing the learning rate linearly for the first $4000$ warmup steps and then decreasing it proportionally to the inverse square root of the step number (page 7).